In [2]:
import os
import sys
sys.path.append('../')
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import json
import pytransform3d.transformations as pt
import pytransform3d.camera as pc
import pytransform3d.visualizer as pv
from data_tools.process_utils import fov2focal, generate_pcd
import torch
from utils.dual_quaternion import quaternion_to_matrix

In [1]:

dataset = 'paris'
subset = 'sapien'
scenes = 'foldchair_102255 washer_103776 fridge_10905 blade_103706 storage_45135 oven_101917 stapler_103111 USB_100109 laptop_10211 scissor_11100'.split(' ')
# subset = 'realscan'
# scenes = 'real_fridge real_storage'.split(' ')
# scene = 'foldchair_102255'

# dataset = 'dta'
# subset = 'sapien'
# scenes = 'fridge_10489 storage_47254'.split(' ')
scene = 'fridge_10489'

root = "/home/yuliu/Dataset/ArtGS_raw_data"
dataset = 'artgs'
subset = 'sapien'
scenes = 'oven_101908 table_25493 storage_45503 storage_47648 table_31249'.split(' ')
scene = 'table_25493'

filename = f'{root}/{dataset}/{subset}/{scene}/transforms_train.json'
# filename = '/mnt/fillipo/yuliu/VideoArtGS/ruijie_dying/transforms_train.json'
info = json.load(open(filename, 'r'))

poses = [np.array(frame['transform_matrix'], np.float32) for frame in info['frames']]
vis_poses = poses


w, h = Image.open(f'{root}/{dataset}/{subset}/{scene}/start/train/rgba/0000.png').size
# w, h = 384, 384
fovx, fovy = info['camera_angle_x'], info['camera_angle_y']
K = np.zeros((3, 3))
K[0, 0] = fov2focal(fovx, w)
K[1, 1] = fov2focal(fovy, h)
K[0, 2] = w // 2
K[1, 2] = h // 2
K[2, 2] = 1
sensor_size = (float(w), float(h))

fig = pv.figure()
mesh_filename = f'{root}/{dataset}/{subset}/{scene}/gt/start/start_rotate.ply'
# mesh_filename = '/mnt/fillipo/yuliu/VideoArtGS/ruijie_dying/base_part.ply'
fig.plot_mesh(mesh_filename)

ill_poses = []
for i, pose in enumerate(vis_poses):
    try:
        fig.plot_transform(A2B=pose, s=0.1)
        fig.plot_camera(M=K, cam2world=pose, virtual_image_distance=0.1, sensor_size=sensor_size)
    except:
        ill_poses.append((i, pose))
fig.show()

NameError: name 'json' is not defined

In [3]:
root = "../data"

dataset = 'v2a'
subset = 'sapien'

scene = '100068_joint_0_bg_view_0'

filename = f'{root}/{dataset}/{subset}/{scene}/transforms.json'
info = json.load(open(filename, 'r'))

poses = [np.array(frame['transform_matrix'], np.float32) for frame in info['frames']]
vis_poses = poses


w, h = Image.open(f'{root}/{dataset}/{subset}/{scene}/images/000000.png').size
# w, h = 384, 384
fovx, fovy = info['camera_angle_x'], info['camera_angle_y']
K = np.zeros((3, 3))
K[0, 0] = fov2focal(fovx, w)
K[1, 1] = fov2focal(fovy, h)
K[0, 2] = w // 2
K[1, 2] = h // 2
K[2, 2] = 1
sensor_size = (float(w), float(h))

fig = pv.figure()
mesh_filename = f'{root}/{dataset}/{subset}/{scene}/gt/whole_mesh.ply'
# mesh_filename = '/mnt/fillipo/yuliu/VideoArtGS/ruijie_dying/base_part.ply'
fig.plot_mesh(mesh_filename)

ill_poses = []
for i, pose in enumerate(vis_poses):
    try:
        fig.plot_transform(A2B=pose, s=0.1)
        fig.plot_camera(M=K, cam2world=pose, virtual_image_distance=0.1, sensor_size=sensor_size)
    except:
        ill_poses.append((i, pose))
fig.show()
print("ill poses", len(ill_poses))

ill poses 0


In [4]:
import open3d as o3d
from PIL import Image


def pcd_to_mesh(pcd_path, output_path, voxel_size=0.05):
    """
    Convert point cloud to mesh using Open3D
    
    Args:
        pcd_path (str): Path to input PCD file
        output_path (str): Path to save output mesh file
        voxel_size (float): Voxel size for downsampling
    """
    pcd = o3d.io.read_point_cloud(pcd_path)
    downpcd = pcd.voxel_down_sample(voxel_size)
    downpcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        downpcd, depth=5, width=0, scale=1.1, linear_fit=False)[0]
    # vertices_to_remove = mesh.compute_vertex_normals()
    # mesh.remove_vertices_by_mask(vertices_to_remove)
    o3d.io.write_triangle_mesh(output_path, mesh)


data_path = '/mnt/fillipo/yuliu/video2articulation/sim_data/partnet_mobility/Microwave/7265/joint_0_bg/view_0'
poses_raw = np.load(f'{data_path}/camera_pose.npy')
poses = []
for p in poses_raw:
    rot = quaternion_to_matrix(torch.from_numpy(p[3:]).float()).numpy()
    rot = rot[:, [1, 2, 0]] * [[-1, 1, -1]]
    t = p[:3]
    pose = np.eye(4)
    pose[:3, :3] = rot
    pose[:3, 3] = t
    poses.append(pose)

K = np.load(f'{data_path}/intrinsics.npy')
w, h = K[0, 2] * 2, K[1, 2] * 2
sensor_size = (float(w), float(h))

cano_path = '/mnt/fillipo/yuliu/video2articulation/sim_data/partnet_mobility/Microwave/7265/joint_0_bg/view_init'
cano_poses = np.load(os.path.join(cano_path, 'camera_pose.npy'))
vis_poses = np.concatenate([poses, cano_poses], axis=0)
# cano_pcd_paths = os.listdir(os.path.join(cano_path, 'xyz'))
# cano_xyzs = []
# cano_colors = []
# for pcd_path in cano_pcd_paths:
#     seg = np.load(os.path.join(cano_path, 'segment', pcd_path))['a'].reshape(-1)
#     xyz = np.load(os.path.join(cano_path, 'xyz', pcd_path))['a']
#     color = np.array(Image.open(os.path.join(cano_path, 'rgb', pcd_path).replace('.npz', '.png')))
#     alpha = color[:, :, 3].reshape(-1)
#     color = color[:, :, :3].reshape(-1, 3)
#     xyz, color = xyz[alpha > 0], color[alpha > 0]
#     cano_xyzs.append(xyz)
#     cano_colors.append(color / 255)
# cano_xyzs = np.concatenate(cano_xyzs, axis=0)
# cano_colors = np.concatenate(cano_colors, axis=0)
# cano_pcd = o3d.geometry.PointCloud()
# cano_pcd.points = o3d.utility.Vector3dVector(cano_xyzs)
# cano_pcd.colors = o3d.utility.Vector3dVector(cano_colors)
# cano_pcd = cano_pcd.voxel_down_sample(voxel_size=0.05)
# o3d.io.write_point_cloud(os.path.join(cano_path, 'cano_pcd.ply'), cano_pcd)
cano_pcd = o3d.io.read_point_cloud(os.path.join(cano_path, 'cano_pcd.ply'))
xyz = np.asarray(cano_pcd.points)
print(xyz.min(axis=0), xyz.max(axis=0))
# # center = np.mean(cano_pcd.points, axis=0)
o3d.visualization.draw_geometries([cano_pcd])
# # mesh = pcd_to_mesh(os.path.join(cano_path, 'cano_pcd.ply'), os.path.join(cano_path, 'cano_mesh.ply'))
# # mesh_filename = os.path.join(cano_path, 'cano_mesh.ply')
# mesh_filename = '/mnt/fillipo/ruijie/dream_art/render_test/microwave/7265/gt/end/end_rotate.ply'
# mesh = o3d.io.read_triangle_mesh(mesh_filename)
# mesh_center = np.mean(mesh.vertices, axis=0)
# mesh.translate(-mesh_center + center)
# o3d.io.write_triangle_mesh(os.path.join(cano_path, 'cano_mesh.ply'), mesh)
mesh_filename = os.path.join(cano_path, 'cano_mesh.ply')
mesh = o3d.io.read_triangle_mesh(mesh_filename)
o3d.visualization.draw_geometries([mesh, cano_pcd])
fig = pv.figure()
fig.plot_mesh(mesh_filename)

ill_poses = []
for i, pose in enumerate(vis_poses):
    try:
        fig.plot_transform(A2B=pose, s=0.1)
        fig.plot_camera(M=K, cam2world=pose, virtual_image_distance=0.1, sensor_size=sensor_size)
    except:
        ill_poses.append((i, pose))
fig.show()

[-9.50000191e+00 -9.50000191e+00 -9.53674316e-07] [9.50000191 9.50000072 6.74819994]


In [3]:
print("ill poses", ill_poses)

ill poses []
